# Rotterdam LST, NDVI, and LCZ calculation

Prerequsites:

```
pip install geemap ee geopandas rasterio rasterstats
```

## Initialise the Project

In [5]:
import os
import ee
import geemap
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats

# ==========================================
# STEP 1: INITIALIZE EARTH ENGINE
# ==========================================

try:
    ee.Initialize(project='applied-spatial-rotterdam')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='applied-spatial-rotterdam')

# ==========================================
# STEP 2: DEFINE AOI & PROCESSING FUNCTIONS
# ==========================================

rotterdam_aoi = ee.Geometry.Rectangle([3.9315, 51.8176, 4.7, 52.0125])

def mask_landsat_clouds(image):
    """Mask clouds and cloud shadows using QA_PIXEL band (bits 3 and 4)."""
    qa = image.select('QA_PIXEL')
    mask = qa.bitwiseAnd(1 << 4).eq(0).And(qa.bitwiseAnd(1 << 3).eq(0))
    return image.updateMask(mask)

def process_landsat_indicators(image):
    """Scale optical/thermal bands and derive LST (Celsius) and NDVI."""
    optical = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    ndvi = optical.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    lst_celsius = (
        image.select('ST_B10')
        .multiply(0.00341802).add(149.0).subtract(273.15)
        .rename('LST_Celsius')
    )
    return (
        image
        .addBands(optical, None, True)
        .addBands(lst_celsius, None, True)
        .addBands(ndvi)
    )

def normalize_lst(image):
    """Subtract per-image spatial mean to produce LST anomaly (normalized LST)."""
    lst = image.select('LST_Celsius')
    image_mean = lst.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=rotterdam_aoi,
        scale=30,
        maxPixels=1e9
    ).getNumber('LST_Celsius')
    return (
        lst.subtract(image_mean)
        .rename('LST_Normalized')
        .toFloat()
        .copyProperties(image, image.propertyNames())
    )

## Build Landsat Composite

In [6]:
# ==========================================
# STEP 3: BUILD LANDSAT COMPOSITE
# ==========================================

landsat_collection = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterBounds(rotterdam_aoi)
    .filterDate('2025-05-01', '2025-08-31')
    .filter(ee.Filter.lt('CLOUD_COVER', 30))
    .map(mask_landsat_clouds)
    .map(process_landsat_indicators)
)

print(f"Scenes found for summer 2025: {landsat_collection.size().getInfo()}")

# Normalized LST composite (anomaly relative to each image's spatial mean)
lst_normalized = (
    landsat_collection
    .map(normalize_lst)
    .select('LST_Normalized')
    .median()
    .clip(rotterdam_aoi)
)

# Median composites for NDVI and raw LST
summer_composite = landsat_collection.median().clip(rotterdam_aoi)
ndvi_layer = summer_composite.select('NDVI')

print("Finished building composite")

Scenes found for summer 2025: 6


## Extract LCZ

In [7]:
# ==========================================
# STEP 4: EXTRACT LOCAL CLIMATE ZONES (LCZ)
# ==========================================

print("Extracting Local Climate Zones...")

lcz_layer = (
    ee.ImageCollection('RUB/RUBCLIM/LCZ/global_lcz_map/latest')
    .mosaic()
    .clip(rotterdam_aoi)
    .select('LCZ_Filter')
)

print("Done")

Extracting Local Climate Zones...


## Write Rasters

In [9]:
# ==========================================
# STEP 5: EXPORT RASTERS
# ==========================================

os.makedirs('../data/rotterdam', exist_ok=True)

print("Exporting normalized LST raster (30m)...")
geemap.ee_export_image(
    lst_normalized,
    filename='../data/rotterdam/rotterdam_lst_normalized_2025.tif',
    scale=30,
    region=rotterdam_aoi,
    file_per_band=False
)

print("Exporting NDVI raster (30m)...")
geemap.ee_export_image(
    ndvi_layer,
    filename='../data/rotterdam/rotterdam_ndvi_2025.tif',
    scale=30,
    region=rotterdam_aoi,
    file_per_band=False
)

print("Exporting LCZ raster (100m)...")
geemap.ee_export_image(
    lcz_layer,
    filename='../data/rotterdam/rotterdam_lcz_2018.tif',
    scale=100,
    region=rotterdam_aoi,
    file_per_band=False
)

print("All rasters exported to ../data/rotterdam/")

Exporting normalized LST raster (30m)...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\artem\Documents\TUDelft\ARFW0501\report\data\rotterdam\rotterdam_lst_normalized_2025.tif
Exporting NDVI raster (30m)...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\artem\Documents\TUDelft\ARFW0501\report\data\rotterdam\rotterdam_ndvi_2025.tif
Exporting LCZ raster (100m)...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\artem\Documents\TUDelft\ARFW0501\report\data\rotterdam\rotterdam_lcz_2018.tif
All rasters exported to ../data/rotterdam/


## Enrich GeoPackage with Raster Data

In [10]:
# ==========================================
# STEP 6: ZONAL STATISTICS ON GEOPACKAGE
# ==========================================

gpkg_path    = "../data/rotterdam/wijkenbuurten_2024.gpkg"
lst_raster   = "../data/rotterdam/rotterdam_lst_normalized_2025.tif"
ndvi_raster  = "../data/rotterdam/rotterdam_ndvi_2025.tif"
lcz_raster   = "../data/rotterdam/rotterdam_lcz_2018.tif"
output_gpkg  = "../data/rotterdam/rotterdam_wijkenbuurten_enriched.gpkg"

print("\nLoading GeoPackage layers...")
buurten_gdf = gpd.read_file(gpkg_path, layer="buurten")

# Keep only buurten belonging to the municipality of Rotterdam
buurten_gdf = buurten_gdf[buurten_gdf['gemeentenaam'] == 'Rotterdam'].copy()
print(f"  Filtered to {len(buurten_gdf)} buurten in Rotterdam")

# Reproject vectors to match raster CRS if needed
with rasterio.open(lst_raster) as src:
    raster_crs = src.crs

if buurten_gdf.crs != raster_crs:
    print(f"Reprojecting vectors to raster CRS: {raster_crs}")
    buurten_gdf = buurten_gdf.to_crs(raster_crs)

def extract_zonal_metrics(gdf, raster_path, stat, column_name):
    """Compute a zonal statistic from a raster for each polygon in a GeoDataFrame."""
    print(f"  {column_name} <- {stat}({os.path.basename(raster_path)})")
    stats = zonal_stats(gdf, raster_path, stats=[stat])
    gdf[column_name] = [x[stat] if x else None for x in stats]
    return gdf

print("\nProcessing Buurten (Neighborhoods)...")
buurten_gdf = extract_zonal_metrics(buurten_gdf, lst_raster,  'mean',     'mean_LST_normalized')
buurten_gdf = extract_zonal_metrics(buurten_gdf, ndvi_raster, 'mean',     'mean_NDVI')
buurten_gdf = extract_zonal_metrics(buurten_gdf, lcz_raster,  'majority', 'majority_LCZ')

print(f"\nSaving enriched layer to {output_gpkg}...")
buurten_gdf.to_file(output_gpkg, layer="buurten_enriched", driver="GPKG")

print("Done! Output files:")
print(f"  ../data/rotterdam/rotterdam_lst_normalized_2025.tif")
print(f"  ../data/rotterdam/rotterdam_ndvi_2025.tif")
print(f"  ../data/rotterdam/rotterdam_lcz_2018.tif")
print(f"  {output_gpkg}  (layer: buurten_enriched)")


Loading GeoPackage layers...
  Filtered to 92 buurten in Rotterdam
Reprojecting vectors to raster CRS: EPSG:4326

Processing Buurten (Neighborhoods)...
  mean_LST_normalized <- mean(rotterdam_lst_normalized_2025.tif)


C:\Users\artem\AppData\Local\Programs\Python\Python313\Lib\site-packages\rasterstats\io.py:437: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(


  mean_NDVI <- mean(rotterdam_ndvi_2025.tif)
  majority_LCZ <- majority(rotterdam_lcz_2018.tif)

Saving enriched layer to ../data/rotterdam/rotterdam_wijkenbuurten_enriched.gpkg...
Done! Output files:
  ../data/processed/rotterdam_lst_normalized_2025.tif
  ../data/processed/rotterdam_ndvi_2025.tif
  ../data/processed/rotterdam_lcz_2018.tif
  ../data/rotterdam/rotterdam_wijkenbuurten_enriched.gpkg  (layer: buurten_enriched)
